In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
sys.path.append("../src")
from pyspark.sql import functions as F
import observability.metrics 
import observability.runner
import observability.config
import observability.anomalies
import observability.anomalies_runner
import importlib
importlib.reload(observability.metrics)
importlib.reload(observability.runner)
importlib.reload(observability.config)
importlib.reload(observability.anomalies)
importlib.reload(observability.anomalies_runner)
from observability.metrics import RowCountMetric, NullRateMetric, DistinctCountMetric
from observability.runner import ObservabilityRunner
import time
import statistics
import datetime
import uuid
from observability.config import METRIC_HISTORY_TABLE 
from observability.anomalies import RowCountAnomalyDetector, NullRateAnomalyDetector
from observability.anomalies_runner import AnomalyDetectorRunner


In [0]:

metrics_config = [
    {
        "metric": "row_count",
        "enabled": True
    },
    {
        "metric": "null_rate",
        "enabled": True,
        "column_name": "amount"
    },
    {
        "metric": "distinct_count",
        "enabled": True,
        "column_name": "country"
    }
]

metric_classes = {
    "row_count": RowCountMetric,
    "null_rate": NullRateMetric,
    "distinct_count": DistinctCountMetric
}

metrics = []

for item in metrics_config:

    if not item["enabled"]:
        continue
    elif item["enabled"] and "column_name" not in item:
        metric_name = metric_classes[item["metric"]]()
        metrics.append(metric_name)
    elif item["enabled"] and "column_name" in item:
        metric_name = metric_classes[item["metric"]](item["column_name"])
        metrics.append(metric_name)

ANOMALY_THRESHOLD = {
    "row_count": 30, # (percent)
    "null_rate": 10, # max null rate (percent)
}

anomalies = [
    RowCountAnomalyDetector,
    NullRateAnomalyDetector
]

anomalies_type = {
    RowCountAnomalyDetector: "row_count",
    NullRateAnomalyDetector: "null_rate"
}
        

In [0]:
orders_df = (
            spark.range(1_050_000)
            .withColumnRenamed("id", "order_id")
            .withColumn(
                "country",
                F.when(F.col("order_id") % 10 == 0, None)
                .when(F.col("order_id") % 3 == 0, "PL")
                .when(F.col("order_id") % 3 == 1, "DE")
                .otherwise("UK")
                )
            .withColumn(
                "amount",
                F.when(F.col("order_id") % 3 == 0, None)
                .otherwise((F.rand() * 1000).cast("double"))
            )
            )

display(orders_df)
orders_df.printSchema()
orders_df.select(F.spark_partition_id().alias("partition_id")).groupBy("partition_id").count().orderBy("partition_id").show()

In [0]:
# v1_times = []

# for i in range(0, 20):

#     observability_runner = ObservabilityRunner(metrics, orders_df)

#     start = time.perf_counter()

#     results =  observability_runner.run()

#     end = time.perf_counter()

#     baseline_time_v1 = end-start

#     v1_times.append(baseline_time_v1)

# print(f"Baseline runtimes: {v1_times} seconds")

In [0]:
table_name = "orders"

observability_runner_v2 = ObservabilityRunner(metrics, orders_df)

start=time.perf_counter()

run_result = observability_runner_v2.run().collect()

end=time.perf_counter()

run_result = run_result[0].asDict()

run_id = str(uuid.uuid4())

timestamp = datetime.datetime.now()

result_time = end-start

result = {
    "run_id": run_id,
    "table_name": table_name,
    **run_result,
    "executed_at": timestamp
}

ddl_schema = "run_id STRING NOT NULL, table_name STRING, row_count LONG, null_rate_amount DOUBLE, distinct_count_country INT, executed_at TIMESTAMP"

result_df = (
    spark.createDataFrame([result], 
                          schema=ddl_schema)
    )

display(result_df)

result_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(METRIC_HISTORY_TABLE)

In [0]:
# orders_df.count()
# orders_df.select("country").distinct().count()
# orders_df.select("country").distinct().explain("formatted")

In [0]:
# v2_times = []

# for i in range (0, 20):

#     start = time.perf_counter()
#     optimized_df = orders_df.agg(
#         F.count("*").alias("row_count"),
#         (F.sum(
#             F.when(F.col("amount").isNull(), 1).otherwise(0)
#         ) / F.count("*")).alias("null_rate"),
#         F.countDistinct("country").alias("distinct_country_count")
#     )
#     result = optimized_df.collect()
#     end = time.perf_counter()

#     baseline_time_v2 = end-start

#     v2_times.append(baseline_time_v2)

# print(f"Baseline runtime after optimization: {v2_times} seconds")

# # optimized_df.explain("formatted")

In [0]:
# medians

# print(f"v1_times median: {statistics.median(v1_times)}")
# print(f"v2_times median: {statistics.median(v2_times)}")

In [0]:
anomaly_run = AnomalyDetectorRunner(spark, METRIC_HISTORY_TABLE, anomalies, anomalies_type, ANOMALY_THRESHOLD)
anomaly_result = anomaly_run.run_anomaly_detector()
anomaly_result